***Welcome! Hello!***

We are glad you're here 👏! And are excited to get you rolling 🛣!

This notebook exists to enable you scan J1939.

It builds on the things done in `easy_*.ipynb` so if you haven't used those yet, please start there.

This is not an 'easy' notebook so we may ask you to edit the code in the cells.

First we'll do some imports and define some functions we will use later

In [ ]:
import getopt
import sys
import signal
import re
import threading
import time

from IPython.core.interactiveshell import InteractiveShell
from scapy.contrib.automotive.scanner.enumerator import ServiceEnumerator

InteractiveShell.ast_node_interactivity = 'all'

import ipywidgets as widgets
from ipywidgets import interact, interact_manual, Textarea

import binascii
import pandas as pd
import qgrid

qgrid.enable()

In [ ]:
from scapy.all import *

load_layer("can")
conf.contribs['CANSocket'] = {'use-python-can': True}
load_contrib('cansocket')
load_contrib('isotp')
load_contrib('automotive.uds')
load_contrib('automotive.uds_scan')

In [ ]:
from tabulate import tabulate

def generate_markdown_table(sa_dict):
    """
    Generates a Markdown table using the tabulate library.
    Ingests: {SA_INT: {'methods': [...], 'packet': ...}}
    """
    # 1. Identify all unique methods for columns
    all_methods = set()
    for data in sa_dict.values():
        all_methods.update(data.get('methods', []))
    
    sorted_methods = sorted(list(all_methods))
    
    # 2. Prepare the table headers
    headers = ["SA"] + sorted_methods
    
    # 3. Prepare the rows
    table_data = []
    # Sorting by integer SA key to keep the table organized
    for sa in sorted(sa_dict.keys()):
        row = [f"0x{sa:02X}"] # Format as hex (e.g., 0x2A)
        found_methods = sa_dict[sa].get('methods', [])
        
        for method in sorted_methods:
            row.append("X" if method in found_methods else "")
        
        table_data.append(row)
    
    # 4. Generate the markdown string
    return tabulate(table_data, headers=headers, tablefmt="github")

In [ ]:
load_contrib('automotive.j1939')
from scapy.contrib.cansocket import CANSocket
from scapy.contrib.automotive.j1939.j1939_scanner import j1939_scan, j1939_scan_passive
from scapy.contrib.automotive.j1939.j1939_dm_scanner import j1939_scan_dm

If you want to see more details of what's going on during the scans, uncomment the below

In [ ]:
import logging
from scapy.contrib.automotive.j1939.j1939_soft_socket import log_j1939

# Set the level to DEBUG
# log_j1939.setLevel(logging.DEBUG)

# Do the thing: Scan J1939

Let's define the python-can INTERFACE, CHANNEL, and BITRATE we will be using

In [ ]:
INTERFACE='candle'
CHANNEL='0'
BITRATE=500000

Now we will scan for the current CA (Controller Applications) passively; i.e. by just listening for traffic and seeing what is received.

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    found = j1939_scan_passive(csock)

#for sa, info in found.items():
for sa in found:
    print("SA=0x{:02X}  found_by=passive".format(sa))

print(f"found {len(found)}")

We'll save those CAs we found as `found_passive`

In [ ]:
found_passive = found

## Active Scans

Then we will scan using 4 of the active methods (all except UDS method which we will use later)

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    found = j1939_scan(csock, force=True, methods=['addr_claim', 'ecu_id', 'rts_probe', 'unicast'])

for sa, info in found.items():
    print("SA=0x{:02X}  found_by={}".format(sa, info['methods']))

print(generate_markdown_table(found))

We'll save these CAs that we just found using active scan techniques as `found_active`

In [ ]:
found_active = found.keys()

## Identify UDS ECUs

Let's use a 'UDS' scanning technique. This will start with the assumption that there is a UDS server on PGN 0xDA at the CA address, send a "Tester Present" request and listen for responses. If a response is received then the assumption was probably correct and we know 1) there is a CA at that address and 2) the CA supports UDS on PGN 0xDA.

The scan tries both 'functional addressing' (a broadcast tester present request) and 'physical addressing' (a unicast tester present request).

In [ ]:
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    found = j1939_scan(csock, force=True, methods=['uds'])

for sa, info in found.items():
    print("SA=0x{:02X}  found_by={}".format(sa, info['methods']))


Finally, let's save these CAs as `found_uds`

In [ ]:
found_uds=found.keys()

## DM Scanning Each

Another J1939-specific thing is DMs. We can test each of those identified CA addresses for the DMs that they support

In [ ]:
def print_dm_matrix(scan_data):
    """
    Prints a wide table where each row is a Source Address and each column is a DM.
    """
    if not scan_data:
        print("No scan data provided.")
        return

    # 1. Identify all unique DM names present across all SAs and sort them naturally
    all_dm_names = set()
    for dms in scan_data.values():
        all_dm_names.update(dms.keys())
    
    # Natural sort: DM1, DM2 ... DM10 instead of DM1, DM10, DM2
    sorted_columns = sorted(list(all_dm_names), 
                            key=lambda x: int(x.replace('DM', '')) if x.startswith('DM') else 0)

    # 2. Define column width based on the longest error message or DM name
    col_width = 10 
    sa_width = 6

    # 3. Print Header
    header = f"{'SA':<{sa_width}} | " + " | ".join(f"{dm:<{col_width}}" for dm in sorted_columns)
    print(header)
    print("-" * len(header))

    # 4. Print Rows
    for sa in sorted(scan_data.keys()):
        row_cells = []
        for dm in sorted_columns:
            result = scan_data[sa].get(dm)
            
            if result is None:
                cell_text = "-" # DM not probed for this SA
            elif result.supported:
                cell_text = "✓"
            else:
                # Use error string (Timeout, NACK, etc.)
                cell_text = result.error if result.error else "No"
            
            row_cells.append(f"{cell_text:<{col_width}}")
        
        print(f"{sa:<{sa_width}} | " + " | ".join(row_cells))

# Example Usage:
# print_dm_matrix(your_scan_results)

def print_dm_matrix_transposed(scan_data):
    """
    Prints a transposed DM support table:
    Rows = DM names (DM1, DM2, etc.)
    Columns = Source Addresses (19, 20, etc.)
    """
    if not scan_data:
        print("No scan data provided.")
        return

    # 1. Get and sort SAs (Columns)
    sorted_sas = sorted(scan_data.keys())
    
    # 2. Get and sort all unique DM names (Rows)
    all_dm_names = set()
    for dms in scan_data.values():
        all_dm_names.update(dms.keys())
    
    sorted_dms = sorted(list(all_dm_names), 
                        key=lambda x: int(x.replace('DM', '')) if x.startswith('DM') else 0)

    # 3. Define widths
    dm_col_width = 6
    sa_col_width = 12  # Wide enough for "Timeout" or "NACK"

    # 4. Print Header (SA IDs)
    header = f"{'DM':<{dm_col_width}} | " + " | ".join(f"SA {sa:<{sa_col_width-3}}" for sa in sorted_sas)
    print(header)
    print("-" * len(header))

    # 5. Print Rows (One row per DM)
    for dm in sorted_dms:
        row_cells = []
        for sa in sorted_sas:
            result = scan_data[sa].get(dm)
            
            if result is None:
                cell_text = "-"
            elif result.supported:
                cell_text = "✓"
            else:
                # Combine supported/error into the cell as requested
                cell_text = result.error if result.error else "No"
            
            row_cells.append(f"{cell_text:<{sa_col_width}}")
        
        print(f"{dm:<{dm_col_width}} | " + " | ".join(row_cells))

# Example usage:
# print_dm_matrix_transposed(your_scan_results)

In [ ]:
target_cas = set().union(found_passive, found_active, found_uds)
results = {}
with CANSocket(interface=INTERFACE, channel=CHANNEL, bitrate=BITRATE) as csock:
    for da in target_cas:
        results[da] = j1939_scan_dm(csock, target_da=da, dms=["DM1", "DM14"])

print_dm_matrix(results)